In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error




file_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(file_path)


In [ ]:
# Task 2: Write your code here:
print(df.head())

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
print(df.describe())

In [ ]:
# Task 1: Write your code here:
#cheking missing values
missing_info = df.isnull().sum()
print(missing_info[missing_info > 0])

#handle
df = df.fillna(df.median(numeric_only=True))
df = df.apply(lambda x: x.fillna(x.value_counts().index[0]) if x.dtype == "object" else x)

In [ ]:
# Task 2: Write your code here:
#checking
duplicates_count = df.duplicated().sum()
print(f"Number of duplicates: {duplicates_count}")

#removing duplicates
df = df.drop_duplicates()

In [ ]:
# Task 3: Write your code here:
#identifying
cat_cols = df.select_dtypes(include=['object', 'category']).columns

#one hot encoding
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

#scaling all features
features = df.columns
df[features] = scaler.fit_transform(df[features])

In [ ]:
# Task 5: Write your code here:
#replace
target_col = 'Target'
imbalance_ratio = df[target_col].value_counts(normalize=True)
print(f"Target Distribution:\n{imbalance_ratio}")

is_imbalanced = imbalance_ratio.min() < 0.2
print(f"Data is {'IMBALANCED' if is_imbalanced else 'BALANCED'}")



In [ ]:
# Task 1: Write your code here:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm imbalanced-learn -q

clear_output()

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from catboost import CatBoostClassifier
import numpy as np



X = df.drop('Target', axis=1)
y = df['Target']




In [ ]:
# Task 2,3,4,5: Write your code here:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    BoostCatModel = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        verbose=0,
        eval_metric='F1'
    )

    BoostCatModel.fit(X_train, y_train)

    y_pred = BoostCatModel.predict(X_val)
    f1_scores.append(f1_score(y_val, y_pred))

print(f"Average F1 Score across 5 folds: {np.mean(f1_scores):.4f}")

In [ ]:
# Task 1: Write your code here:

import pandas as pd

importances = BoostCatModel.get_feature_importance()
feature_names = X.columns

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

golden_feature_name = feature_importance_df.iloc[0]['Feature']
golden_feature_score = feature_importance_df.iloc[0]['Importance']

print(f"The Golden Feature is: {golden_feature_name}")
print(f"Confidence Score: {golden_feature_score:.2f}")

In [ ]:
# Task 2: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_importance(df):
    plt.figure(figsize=(12, 8))

    sns.barplot(
        x='Importance',
        y='Feature',
        data=df.head(20),
        palette='viridis'
    )

    plt.title('Top 20 Features Driving Credit Default Prediction', fontsize=16)
    plt.xlabel('Importance Score (CatBoost Units)', fontsize=12)
    plt.ylabel('Anonymized Feature Name', fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)

    plt.gca().get_yticklabels()[0].set_color('gold')
    plt.gca().get_yticklabels()[0].set_weight('bold')

    plt.tight_layout()
    plt.savefig('feature_importance.png')
    plt.show()

plot_importance(feature_importance_df)

In [ ]:
# Task Bonus: Write your code here:
import numpy as np
from sklearn.metrics import accuracy_score # Per prompt request for comparison

X_golden = X_scaled[[golden_feature_name]]

golden_scores = []
full_model_scores = []

print(f"Retraining with ONLY: {golden_feature_name}...")

for train_idx, val_idx in skf.split(X_golden, y):
    X_train_g, X_val_g = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_golden = CatBoostClassifier(iterations=500, verbose=0, random_seed=42)
    model_golden.fit(X_train_g, y_train)

    y_pred = model_golden.predict(X_val_g)
    golden_scores.append(accuracy_score(y_val, y_pred))

avg_golden_acc = np.mean(golden_scores)

print("-" * 30)
print(f"Full Model Accuracy:    {avg_full_acc:.4f}")
print(f"Golden Feature Only:    {avg_golden_acc:.4f}")
print(f"Performance Retained:   {(avg_golden_acc / avg_full_acc)*100:.2f}%")